# resnet-stem — ex2: contrast ResNet stem with a single Conv 3×3 stride 2 alternative

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `resnet-stem`. Running the final beacon cell reports progress against the `CNN: ResNet stem block` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
import torch.nn as nn
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: ResNet stem block` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`resnet-stem`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "resnet-stem"
DD_SUBTOPIC = "CNN: ResNet stem block"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Stem block vs single-conv downsample — output shape contrast

Ex1 BUILT the 4-op ResNet stem (Conv 7×7 s=2 → BN → ReLU → MaxPool 3×3 s=2) and verified `(B, 3, 224, 224) → (B, 64, 56, 56)`. The deepening move is to compare against a NAIVE alternative — a single Conv 3×3 stride 2 — and show the output shapes diverge.

**Naive alternative:** `nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1)`. One operation. Halves spatial dims once.

Shape math (input 224×224):
- ResNet stem: 224 → 112 (Conv s=2) → 56 (MaxPool s=2). **Final: 56×56.**
- Naive 3×3 s=2: `(224 + 2 - 3) // 2 + 1 = 112`. **Final: 112×112.**

**Why ResNet does TWO downsamples in the stem.** A 4× total reduction up front cuts downstream FLOPs by 16× (in the spatial axes). The cost is information loss in the first layers — but the later residual blocks recover representational capacity with depth.

**Receptive field difference.** The 7×7 conv sees a much larger input patch per output pixel than 3×3. Combined with the MaxPool's non-overlapping 2×2 windows (effectively, given stride 2 padding 1), early-layer features encode larger image regions — useful for natural-image tasks but overkill for tiny inputs like CIFAR.

**Channel growth is the same.** Both designs go `3 → 64` — channels are decoupled from the spatial reduction strategy.

### Exercise 2 — contrast ResNet stem with a single Conv 3×3 stride 2 alternative

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the shape and parameter-count contrast between the ResNet stem (Conv 7×7 s=2 + BN + ReLU + MaxPool 3×3 s=2) and a naive single Conv 3×3 stride 2 — same input `(B, 3, 224, 224)` yields different output shapes.
> Keywords: resnet, stem, downsample, shape-math, contrast
> ```

**KCs targeted:** `stem-vs-naive-downsample-shape`, `stem-param-count-vs-naive`

Implement `ex2_compare_stem_vs_naive()`.

Build TWO modules:

1. `stem`: the same `nn.Sequential` as ex1:
   - `nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)`
   - `nn.BatchNorm2d(64)`
   - `nn.ReLU(inplace=True)`
   - `nn.MaxPool2d(kernel_size=3, stride=2, padding=1)`

2. `naive`: a single conv:
   - `nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1)` (default bias=True).

Run a `(1, 3, 224, 224)` zero input through each (use `stem.eval()` and `naive.eval()` first so BatchNorm is deterministic), then return:
```
{
  'stem_out_shape': tuple,         # (1, 64, 56, 56)
  'naive_out_shape': tuple,        # (1, 64, 112, 112)
  'stem_n_params': int,            # sum over stem.parameters()
  'naive_n_params': int,           # sum over naive.parameters()
  'spatial_reduction_factor_stem': int,   # 224 // 56 = 4
  'spatial_reduction_factor_naive': int,  # 224 // 112 = 2
}
```

Constraints:
- `tuple(...)` for shapes, not `torch.Size`.
- The reduction factor is `224 // out_spatial` (integer).

In [ ]:
def ex2_compare_stem_vs_naive() -> dict:
    """Compare ResNet stem (4x downsample) vs naive 3x3 s=2 (2x downsample)."""
    raise NotImplementedError()


def _test_ex2():
    rep = ex2_compare_stem_vs_naive()

    # === Required keys present ===
    needed = {'stem_out_shape', 'naive_out_shape', 'stem_n_params', 'naive_n_params',
              'spatial_reduction_factor_stem', 'spatial_reduction_factor_naive'}
    assert set(rep.keys()) >= needed, f'missing keys: {needed - set(rep.keys())}'

    # === Stem output shape: (1, 64, 56, 56) ===
    assert rep['stem_out_shape'] == (1, 64, 56, 56), f'stem shape wrong: {rep["stem_out_shape"]}'

    # === Naive output shape: (1, 64, 112, 112) ===
    assert rep['naive_out_shape'] == (1, 64, 112, 112), f'naive shape wrong: {rep["naive_out_shape"]}'

    # === Reduction factors: stem 4x, naive 2x ===
    assert rep['spatial_reduction_factor_stem'] == 4, f'stem 4x reduction, got {rep["spatial_reduction_factor_stem"]}x'
    assert rep['spatial_reduction_factor_naive'] == 2, f'naive 2x reduction, got {rep["spatial_reduction_factor_naive"]}x'

    # === Param counts: stem has more (7x7 conv + BN), naive has the 3x3 + bias ===
    # Stem: Conv 7x7 bias=False = 3*64*49 = 9408. BN: 2 * 64 = 128. ReLU/MaxPool: 0. Total 9536.
    assert rep['stem_n_params'] == 9536, f'stem param count: expected 9536, got {rep["stem_n_params"]}'
    # Naive: Conv 3x3 + bias = 3*64*9 + 64 = 1728 + 64 = 1792.
    assert rep['naive_n_params'] == 1792, f'naive param count: expected 1792, got {rep["naive_n_params"]}'

    # === Stem uses MORE params for MORE aggressive downsample (the trade) ===
    assert rep['stem_n_params'] > rep['naive_n_params'], 'stem should be heavier than naive'

    # === Output spatial dims are integer powers of 2 ===
    _, _, hs, ws = rep['stem_out_shape']
    assert hs == ws == 56
    _, _, hn, wn = rep['naive_out_shape']
    assert hn == wn == 112

    # === Stem output has 4x fewer spatial elements than naive ===
    stem_spatial = hs * ws            # 56 * 56 = 3136
    naive_spatial = hn * wn           # 112 * 112 = 12544
    assert naive_spatial // stem_spatial == 4, f'naive has 4x more spatial elements than stem, got {naive_spatial // stem_spatial}x'

    # === Tuples not torch.Size ===
    for k in ('stem_out_shape', 'naive_out_shape'):
        assert type(rep[k]) is tuple, f'{k} must be plain tuple, got {type(rep[k]).__name__}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_compare_stem_vs_naive():
    stem = nn.Sequential(
        nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
        nn.BatchNorm2d(64),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
    )
    naive = nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1)
    stem.eval(); naive.eval()
    x = t.zeros(1, 3, 224, 224)
    with t.no_grad():
        y_stem = stem(x)
        y_naive = naive(x)
    return {
        'stem_out_shape': tuple(y_stem.shape),
        'naive_out_shape': tuple(y_naive.shape),
        'stem_n_params': sum(p.numel() for p in stem.parameters()),
        'naive_n_params': sum(p.numel() for p in naive.parameters()),
        'spatial_reduction_factor_stem': 224 // int(y_stem.shape[-1]),
        'spatial_reduction_factor_naive': 224 // int(y_naive.shape[-1]),
    }
```

**Stem param count math.** Conv 7×7 with `bias=False`: `3 * 64 * 7 * 7 = 9408`. BatchNorm2d(64) has weight + bias = `2 * 64 = 128`. ReLU and MaxPool are parameter-free. Total `9408 + 128 = 9536`.

**Naive param count math.** Conv 3×3 with default `bias=True`: `3 * 64 * 9 + 64 = 1728 + 64 = 1792`. Smaller by ~5×.

**Spatial-element ratio is the FLOPs proxy.** Stem outputs `56² = 3136` per channel; naive outputs `112² = 12544`. A factor-of-4 downstream FLOPs saving — that's why ResNet pays the stem's extra params upfront.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()